# 3 — Evolução do aprendizado e geração por categoria

Leia os resultados de uma **única execução condicional**. Este notebook não treina e não modifica a interface. Escolher uma classe condiciona a geração; não garante que um modelo pouco treinado já a represente corretamente.

In [ ]:
from pathlib import Path
import json, csv, time
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

# Funciona abrindo o Jupyter na raiz ou na pasta notebooks.
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "modelos.py").exists())
CLASSES = ['Camiseta/Top', 'Calça', 'Pulôver', 'Vestido', 'Casaco',
           'Sandália', 'Camisa', 'Tênis', 'Bolsa', 'Bota']
SEMENTE = 42
torch.set_num_threads(2)
torch.manual_seed(SEMENTE)
np.random.seed(SEMENTE)
DISPOSITIVO = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Raiz:', RAIZ, '| Dispositivo:', DISPOSITIVO)


In [ ]:
# A mesma definição é incluída no notebook 03 para permitir leitura independente.
DIM_RUIDO = 100
class GeradorCondicional(nn.Module):
    def __init__(self):
        super().__init__()
        self.classe = nn.Embedding(10, 10)
        self.rede = nn.Sequential(
            nn.ConvTranspose2d(DIM_RUIDO + 10, 128, 7, 1, 0, bias=False),
            nn.BatchNorm2d(128), nn.ReLU(True),
            nn.ConvTranspose2d(128, 64, 4, 2, 1, bias=False),
            nn.BatchNorm2d(64), nn.ReLU(True),
            nn.ConvTranspose2d(64, 1, 4, 2, 1, bias=False), nn.Tanh())

    def forward(self, ruido, rotulos):
        condicao = self.classe(rotulos).view(-1, 10, 1, 1)
        return self.rede(torch.cat([ruido, condicao], dim=1))

class DiscriminadorCondicional(nn.Module):
    def __init__(self):
        super().__init__()
        self.classe = nn.Embedding(10, 28 * 28)
        self.rede = nn.Sequential(
            nn.Conv2d(2, 64, 4, 2, 1, bias=False), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(64, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(128, 1, 7, 1, 0, bias=False))

    def forward(self, imagens, rotulos):
        mapa_classe = self.classe(rotulos).view(-1, 1, 28, 28)
        return self.rede(torch.cat([imagens, mapa_classe], dim=1))


In [ ]:
def painel(imagens, titulo, caminho=None):
    # Dez linhas: uma por classe. Três colunas: três ruídos fixos.
    fig, eixos = plt.subplots(10, 3, figsize=(6, 15))
    for i, eixo in enumerate(eixos.flat):
        eixo.imshow(imagens[i, 0], cmap='gray', vmin=0, vmax=1)
        eixo.set_xticks([]); eixo.set_yticks([])
        if i % 3 == 0:
            eixo.set_ylabel(CLASSES[i // 3], rotation=0, ha='right', va='center')
    fig.suptitle(titulo)
    fig.tight_layout(rect=(0.15, 0, 1, 0.97))
    if caminho is not None:
        fig.savefig(caminho, dpi=120, bbox_inches='tight')
    plt.show()
    plt.close(fig)


In [ ]:
NOME_EXECUCAO = 'verificacao_cgan_notebook'  # mesmo nome usado no notebook 02
PASTA = RAIZ / 'artefatos' / 'cgan' / NOME_EXECUCAO
MODELOS = RAIZ / 'checkpoints' / 'cgan' / NOME_EXECUCAO
assert (MODELOS / 'ultimo.pt').exists(), 'Execute o notebook 02 ou transfira os checkpoints condicionais.'
estado = torch.load(MODELOS / 'ultimo.pt', map_location='cpu', weights_only=True)
assert estado['config']['arquitetura'] == 'cgan_conv_v1'
print('Configuração:', estado['config'])
print('Última época:', estado['epoca'])
if estado['config']['teste_rapido']:
    print('ATENÇÃO: teste de funcionamento; não é evidência de qualidade aprendida.')


## Comparar a evolução com entradas idênticas
Escolhemos início, momento intermediário e último painel disponível. Mesma classe, mesmo ruído e mesma escala. Imagens de outro nome de execução não entram nesta comparação.

In [ ]:
arquivos = sorted(MODELOS.glob('amostras_*.pt'))
assert arquivos, 'Nenhuma amostra salva.'
selecionados = arquivos if len(arquivos) < 3 else [arquivos[0], arquivos[len(arquivos)//2], arquivos[-1]]
if len(selecionados) < 3:
    print('Ainda não existem três momentos disponíveis.')
fig, eixos = plt.subplots(10, len(selecionados), figsize=(3*len(selecionados), 15), squeeze=False)
for coluna, caminho in enumerate(selecionados):
    amostras = torch.load(caminho, map_location='cpu', weights_only=True)
    for classe in range(10):
        eixo = eixos[classe, coluna]
        eixo.imshow(amostras[classe*3, 0], cmap='gray', vmin=0, vmax=1)
        eixo.set_xticks([]); eixo.set_yticks([])
        if classe == 0: eixo.set_title('Época ' + str(int(caminho.stem.split('_')[-1])))
        if coluna == 0: eixo.set_ylabel(CLASSES[classe], rotation=0, ha='right', va='center')
fig.suptitle('cGAN — evolução por categoria' + (' — TESTE DE EXECUÇÃO' if estado['config']['teste_rapido'] else ''))
fig.tight_layout(rect=(0.12, 0, 1, 0.97))
fig.savefig(PASTA / 'comparacao_evolucao.png', dpi=120, bbox_inches='tight'); plt.show(); plt.close(fig)


## Solicitar uma categoria
Altere `CLASSE` e `SEMENTE_GERACAO`. A semente fixa permite explorar o efeito de mudar somente a condição.

In [ ]:
CLASSE = 9  # 0 a 9: consulte a lista abaixo
SEMENTE_GERACAO = 123
assert 0 <= CLASSE < 10
print(dict(enumerate(CLASSES)))
gerador = GeradorCondicional().to(DISPOSITIVO)
gerador.load_state_dict(estado['gerador']); gerador.eval()
rng = torch.Generator().manual_seed(SEMENTE_GERACAO)
ruido = torch.randn(16, DIM_RUIDO, 1, 1, generator=rng).to(DISPOSITIVO)
rotulos = torch.full((16,), CLASSE, dtype=torch.long, device=DISPOSITIVO)
with torch.inference_mode():
    imagens = ((gerador(ruido, rotulos) + 1)/2).clamp(0,1).cpu()
assert imagens.shape == (16,1,28,28) and torch.isfinite(imagens).all()
fig, eixos = plt.subplots(4,4,figsize=(6,6))
for eixo, imagem in zip(eixos.flat, imagens):
    eixo.imshow(imagem[0], cmap='gray', vmin=0, vmax=1); eixo.axis('off')
fig.suptitle(f"Classe solicitada: {CLASSES[CLASSE]} — época {estado['epoca']}")
fig.tight_layout(); plt.show(); plt.close(fig)


## Questões para documentar o aprendizado
1. As saídas lembram a classe solicitada? Em quais categorias há confusão?
2. O mesmo ruído muda de forma coerente ao trocar a classe?
3. Há diversidade dentro de uma categoria ou repetição?
4. Quais mudanças aparecem entre os três momentos?

Escreva observações somente após inspecionar as imagens. Não há classificador auxiliar nem métrica automática de fidelidade à classe. Os testes rápidos comprovam execução, não qualidade. Não invente resultados para a apostila.